# 14 用 PyTorch 高级 API 重写线性回归

本节目标：用 `nn.Linear`、`nn.MSELoss`、`torch.optim.SGD` 和 `DataLoader` 重写线性回归，并理解 API 隐藏了哪些细节。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
torch.set_printoptions(precision=4)

X = torch.randn(1000, 2)
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
y = (X @ true_w + true_b + torch.randn(1000) * 0.01).reshape(-1, 1)

dataset = TensorDataset(X, y)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("X shape:", tuple(X.shape))
print("y shape:", tuple(y.shape))

In [ ]:
model = nn.Sequential(nn.Linear(2, 1))
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

for epoch in range(8):
    for X_batch, y_batch in data_loader:
        y_hat = model(X_batch)
        loss = loss_fn(y_hat, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        train_loss = loss_fn(model(X), y)
    print(f"epoch {epoch + 1}, loss {train_loss.item():.8f}")

learned_w = model[0].weight.data.reshape(-1)
learned_b = model[0].bias.data.item()

print("\n真实 w:", true_w)
print("学到的 w:", learned_w)
print("真实 b:", true_b)
print("学到的 b:", learned_b)

## 从零实现 vs 简洁实现

| 部分 | 从零实现 | PyTorch API 实现 | API 隐藏的细节 |
|---|---|---|---|
| 数据迭代 | 自己写 `data_iter` | `DataLoader` | 打乱、分 batch、迭代 |
| 模型 | 自己写 `X @ w + b` | `nn.Linear` | 参数创建、矩阵乘法、偏置 |
| 损失 | 自己写平方误差 | `nn.MSELoss` | reduction、shape 对齐 |
| 更新 | 手写 `param -= lr * grad` | `torch.optim.SGD` | `no_grad` 更新、参数管理 |
| 清梯度 | 手动 `grad.zero_()` | `optimizer.zero_grad()` | 遍历所有参数并清空梯度 |

简洁实现更适合真实项目；从零实现更适合建立训练循环直觉。